# Rebuild the missing five-minute intraday model bundle

This notebook trains the repository's **34-feature, five-minute `target_30m`** model from the historical seed data already stored in Google Drive. It writes the manifest and both estimators to the exact runtime directory, then verifies hashes and feature contracts. It does **not** deploy Cloud Run or place orders.

Run the cells from top to bottom. Google Drive authorization is required.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Indian_Market_Research')
MODEL_DIR = DRIVE_ROOT / 'models/all_nse_intraday_5mbar_multifactor_replacement_2026'
SEED_DIR = DRIVE_ROOT / 'production_data/nse_official/intraday_seed/olo_minute_archive/symbols'
assert DRIVE_ROOT.is_dir(), f'Missing research root: {DRIVE_ROOT}'
assert SEED_DIR.is_dir(), f'Missing historical seed data: {SEED_DIR}. Stop; do not train from live snapshots alone.'
print('Drive root:', DRIVE_ROOT)
print('Historical seed symbol partitions:', sum(1 for p in SEED_DIR.glob('symbol=*') if p.is_dir()))
print('Output model directory:', MODEL_DIR)


In [ ]:
import subprocess, sys
PROJECT = Path('/content/indian-market-trading')
if not (PROJECT / 'colab_build_all_nse_intraday_multifactor.py').is_file():
    subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/andromyth/indian-market-trading.git',str(PROJECT)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pyarrow','scikit-learn==1.6.1','joblib'], check=True)
print('Repository and dependencies ready:', PROJECT)


In [ ]:
import os
os.chdir(PROJECT)
cmd = [sys.executable, '-u', 'colab_build_all_nse_intraday_multifactor.py', '--drive-root', str(DRIVE_ROOT), '--out', str(MODEL_DIR), '--sample-minutes', '5']
print('Training command:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import hashlib, json, joblib
from all_nse_intraday_model import FEATURES

manifest_path = MODEL_DIR / 'ALL_NSE_INTRADAY_5MBAR_MULTIFACTOR_2026_MODEL_MANIFEST.json'
classifier_path = MODEL_DIR / 'deployment_all_nse_intraday_5mbar_multifactor_classifier_2026.joblib'
regressor_path = MODEL_DIR / 'deployment_all_nse_intraday_5mbar_multifactor_regressor_2026.joblib'
for p in (manifest_path, classifier_path, regressor_path):
    assert p.is_file() and p.stat().st_size > 0, f'Missing/empty artifact: {p}'

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert manifest.get('bar_interval_minutes') == 5, manifest.get('bar_interval_minutes')
assert manifest.get('target') == 'target_30m', manifest.get('target')
assert manifest.get('feature_count') == len(FEATURES) == 34
assert manifest.get('features') == FEATURES, 'Feature order does not match live runtime'
assert manifest.get('classifier_sha256') == sha(classifier_path), 'Classifier hash mismatch'
assert manifest.get('regressor_sha256') == sha(regressor_path), 'Regressor hash mismatch'
clf, reg = joblib.load(classifier_path), joblib.load(regressor_path)
for name, model in [('classifier', clf), ('regressor', reg)]:
    steps = getattr(model, 'named_steps', {})
    imputer = steps.get('imputer') if hasattr(steps, 'get') else None
    n = getattr(imputer, 'n_features_in_', None)
    assert n == 34, f'{name} expects {n} features, not 34'

summary = {
    'result': 'BUNDLE_VERIFIED_LOCALLY_IN_COLAB',
    'model_dir': str(MODEL_DIR),
    'bar_interval_minutes': manifest['bar_interval_minutes'],
    'target': manifest['target'],
    'feature_count': manifest['feature_count'],
    'training_symbol_count': manifest.get('training_symbol_count'),
    'coverage_pct': manifest.get('coverage_pct'),
    'training_rows': manifest.get('training_rows'),
    'holdout_rows': manifest.get('holdout_rows'),
    'validation': manifest.get('validation'),
    'classifier_sha256': manifest['classifier_sha256'],
    'regressor_sha256': manifest['regressor_sha256'],
    'files': {p.name: p.stat().st_size for p in (manifest_path, classifier_path, regressor_path)},
}
print(json.dumps(summary, indent=2))
print('\nIf all checks passed, rerun the GitHub deployment workflow; promotion still requires the live /modelz + Drive write-readiness preflight.')
